In [14]:
import faiss
import numpy as np
import ray
from ray.data import read_parquet

ray.init(ignore_reinit_error=True)


2025-03-20 16:40:37,160	INFO worker.py:1616 -- Calling ray.init() again after it has already been called.


Python version:,3.11.9
Ray version:,2.35.0
Dashboard:,http://127.0.0.1:8266


In [15]:
# Read multiple parquet files into a Ray Dataset directly (no Pandas bottleneck)
dataset = read_parquet("/Users/bikash/stability-ai/version1/image_filter_pipeline/data/interim/part-0.snappy.parquet")

# Check dataset schema
print(dataset.schema())


Parquet Files Sample 0:   0%|          | 0.00/1.00 [00:00<?, ? file/s]

Column              Type
------              ----
image_url           string
face_confidence     double
bbox                string
glasses_label       string
glasses_confidence  double
clip_metadata       string
clip_embedding      list<element: float>


In [16]:
def preprocess_batch(batch):
    embeddings = np.stack(batch["clip_embedding"]).astype("float32")
    embeddings /= np.linalg.norm(embeddings, axis=1, keepdims=True)
    return {"embeddings": embeddings,
            "image_url": batch["image_url"],
            "face_confidence": batch["face_confidence"],
            "bbox": batch["bbox"],
            "glasses_label": batch["glasses_label"]}


# Apply distributed preprocessing
processed_ds = dataset.map_batches(preprocess_batch, batch_format="numpy")

# Check results
processed_ds.show(1)


2025-03-20 16:40:37,568	INFO streaming_executor.py:108 -- Starting execution of Dataset. Full logs are in /tmp/ray/session_2025-03-20_16-27-34_208339_33519/logs/ray-data
2025-03-20 16:40:37,568	INFO streaming_executor.py:109 -- Execution plan of Dataset: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadParquet] -> TaskPoolMapOperator[MapBatches(preprocess_batch)] -> LimitOperator[limit=1]


Running 0: 0.00 row [00:00, ? row/s]

- ReadParquet->SplitBlocks(24) 1: 0.00 row [00:00, ? row/s]

- MapBatches(preprocess_batch) 2: 0.00 row [00:00, ? row/s]

- limit=1 3: 0.00 row [00:00, ? row/s]

{'embeddings': array([ 3.89099936e-03, -2.41540316e-02, -8.55396129e-03,  9.65428073e-03,
        3.84221785e-02, -8.07334948e-03,  1.44045400e-02,  6.46391883e-02,
       -1.81989931e-02, -7.96434178e-04,  3.07101943e-02,  2.91103721e-02,
        2.87889782e-02,  1.76918115e-02,  1.71727035e-02,  1.50795970e-02,
        1.59220666e-01, -1.97044648e-02,  4.05043662e-02, -6.92196330e-03,
       -1.71493050e-02,  1.62084736e-02, -3.18536535e-03, -2.63808831e-03,
       -4.38138470e-03,  2.96347998e-02, -2.11120844e-02, -1.68623198e-02,
        2.75262143e-03,  3.48169729e-02,  2.16280855e-02, -2.54354859e-03,
        1.12521751e-02, -6.94895687e-04, -5.16913943e-02, -9.74996015e-03,
       -2.51179114e-02,  6.30608294e-03,  1.00040548e-02,  1.70883036e-03,
       -1.95561610e-02,  1.23161804e-02,  2.00421158e-02,  8.52543302e-03,
        2.23168507e-02,  2.86801234e-02,  8.71625077e-03, -2.19945982e-02,
        8.88327323e-03,  4.70576249e-03,  4.89773564e-02,  1.66256372e-02,
        1.

In [17]:
# Corrected embedding dimension extraction
sample_embedding = processed_ds.take(1)[0]["embeddings"]
embedding_dim = sample_embedding.shape[0]

print(f"Embedding dimension: {embedding_dim}")

# Create FAISS index (Inner Product)
index = faiss.IndexFlatIP(embedding_dim)


2025-03-20 16:40:37,720	INFO streaming_executor.py:108 -- Starting execution of Dataset. Full logs are in /tmp/ray/session_2025-03-20_16-27-34_208339_33519/logs/ray-data
2025-03-20 16:40:37,721	INFO streaming_executor.py:109 -- Execution plan of Dataset: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadParquet] -> TaskPoolMapOperator[MapBatches(preprocess_batch)] -> LimitOperator[limit=1]


Running 0: 0.00 row [00:00, ? row/s]

- ReadParquet->SplitBlocks(24) 1: 0.00 row [00:00, ? row/s]

- MapBatches(preprocess_batch) 2: 0.00 row [00:00, ? row/s]

- limit=1 3: 0.00 row [00:00, ? row/s]

Embedding dimension: 512


In [18]:
# Add embeddings incrementally to FAISS index (memory efficient)
for batch in processed_ds.iter_batches(batch_size=50_000, batch_format="numpy"):
    embeddings_batch = batch["embeddings"]
    index.add(embeddings_batch)

print(f"Total embeddings indexed: {index.ntotal}")


2025-03-20 16:40:37,914	INFO streaming_executor.py:108 -- Starting execution of Dataset. Full logs are in /tmp/ray/session_2025-03-20_16-27-34_208339_33519/logs/ray-data
2025-03-20 16:40:37,914	INFO streaming_executor.py:109 -- Execution plan of Dataset: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadParquet] -> TaskPoolMapOperator[MapBatches(preprocess_batch)]


Running 0: 0.00 row [00:00, ? row/s]

- ReadParquet->SplitBlocks(24) 1: 0.00 row [00:00, ? row/s]

- MapBatches(preprocess_batch) 2: 0.00 row [00:00, ? row/s]

Total embeddings indexed: 4842


In [19]:
# Search example query (sanity check)
query_embedding = sample_embedding.reshape(1, -1).copy()  # Ensure a writable array
query_embedding /= np.linalg.norm(query_embedding, axis=1, keepdims=True)  # Normalize query

# Perform FAISS search (e.g., find top 5 similar embeddings)
D, I = index.search(query_embedding, k=5)

print("Top-5 nearest indices:", I)
print("Distances:", D)


Top-5 nearest indices: [[   0  520   26  578 2876]]
Distances: [[1.0000001  0.8963572  0.88001055 0.8682898  0.8633883 ]]


In [20]:
faiss.write_index(index, "/Users/bikash/stability-ai/version1/image_filter_pipeline/data/interim/clip_embeddings.index")

In [ ]:
# --- Initialize Ray and FAISS ---
import ray
import faiss
import numpy as np
from ray.data import read_parquet
from transformers import CLIPProcessor, CLIPModel
import torch
import pandas as pd

ray.init(ignore_reinit_error=True)

# --- Load dataset ---
dataset = read_parquet("/Users/bikash/stability-ai/version1/image_filter_pipeline/data/interim/part-0.snappy.parquet")
print("Dataset loaded.")

# --- Load FAISS ---
index = faiss.read_index("/Users/bikash/stability-ai/version1/image_filter_pipeline/data/interim/clip_embeddings.index")
print(f"FAISS index loaded, total entries: {index.ntotal}")

# --- Filter sunglasses labels ---
sunglasses_ds = dataset.filter(lambda x: x["glasses_label"] == "A person wearing sunglasses")
sunglasses_urls = set(sunglasses_ds.to_pandas()["image_url"])
print(f"Sunglasses URLs: {len(sunglasses_urls)} entries")

# --- Generate query embedding (coat) ---
clip_model_name = "openai/clip-vit-base-patch32"
clip_processor = CLIPProcessor.from_pretrained(clip_model_name)
clip_model = CLIPModel.from_pretrained(clip_model_name)

query_text = ["a person wearing a coat"]
with torch.no_grad():
    inputs = clip_processor(text=query_text, return_tensors="pt", padding=True)
    text_embeddings = clip_model.get_text_features(**inputs).numpy()

text_embeddings /= np.linalg.norm(text_embeddings, axis=1, keepdims=True)
print("Query embedding generated.")

# --- Search FAISS index ---
k = 5000
D, I = index.search(text_embeddings, k=k)
matched_indices = I[0]
print(f"FAISS found {len(matched_indices)} matches.")

# --- Get matched URLs from FAISS matches correctly ---
# Retrieve matching URLs via Ray efficiently
matched_urls = dataset.map_batches(
    lambda batch: {"image_url": batch["image_url"]},
    batch_format="numpy"
).take_all()

# matched_indices are positional; get URLs at these positions
matched_urls_df = pd.DataFrame(matched_urls)
faiss_matched_urls = set(matched_urls_df.iloc[matched_indices]["image_url"].tolist())

# --- Intersection of sunglasses + FAISS coat matches ---
final_matches = sunglasses_urls.intersection(faiss_matched_urls)

print(f"\nFound {len(final_matches)} images of people wearing sunglasses and coats:")
for url in final_matches:
    print(url)


2025-03-20 16:47:39,849	INFO worker.py:1774 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8266 


Parquet Files Sample 0:   0%|          | 0.00/1.00 [00:00<?, ? file/s]

2025-03-20 16:47:41,364	INFO streaming_executor.py:108 -- Starting execution of Dataset. Full logs are in /tmp/ray/session_2025-03-20_16-47-38_006077_56180/logs/ray-data
2025-03-20 16:47:41,364	INFO streaming_executor.py:109 -- Execution plan of Dataset: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadParquet] -> TaskPoolMapOperator[Filter(<lambda>)]


Dataset loaded.
FAISS index loaded, total entries: 4842


Running 0: 0.00 row [00:00, ? row/s]

- ReadParquet->SplitBlocks(24) 1: 0.00 row [00:00, ? row/s]

- Filter(<lambda>) 2: 0.00 row [00:00, ? row/s]

Sunglasses URLs: 132 entries
